# Thesis Quality Figures: HySpecNet Mamba and HYPERVIEW2

This notebook prepares the artifact package requested for the thesis. It uses the converted HySpecNet-11k `easy/test` `DATA.npy` archive on Google Drive, selects qualitative examples by reconstruction error quantiles, renders equal-size reconstruction panels and spectra, and creates a HYPERVIEW2 transfer mosaic from saved reconstructions.

Main output directory:

```text
/content/drive/MyDrive/hsi/remote_artifacts/thesis_quality_figures_2026-06-13/
```

The notebook does not modify the thesis repository text. It only writes figures, metadata, and LaTeX snippets for manual inclusion.


## 1. Settings

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/mhx1467/master-thesis-code.git'
REPO_DIR = Path('/content/hsi')
REPO_REF = 'main'

DRIVE_HSI = Path('/content/drive/MyDrive/hsi')

# HySpecNet inputs prepared locally and uploaded to Drive.
HYSPECNET_ARCHIVE = DRIVE_HSI / 'data/archives/hyspecnet_easy_test_data_npy_2026-06-13.tar.zst'
HYSPECNET_ARCHIVE_SHA256 = DRIVE_HSI / 'data/archives/hyspecnet_easy_test_data_npy_2026-06-13.tar.zst.sha256'
HYSPECNET_EXTRACT_PARENT = Path('/content/hsi_data')
HYSPECNET_ROOT = HYSPECNET_EXTRACT_PARENT / 'hyspecnet_easy_test_data_npy'
HYSPECNET_SPLIT = HYSPECNET_ROOT / 'splits/easy/test.csv'
EXPECTED_HYSPECNET_FILES = 1149

CHECKPOINT_PATH = DRIVE_HSI / 'checkpoints/hierarchical_spectral_mamba_ae_k4_spatial_rd_lambda_0_0003_best.pt'

# Artifact package requested in the prompt.
ARTIFACT_ROOT = DRIVE_HSI / 'remote_artifacts/thesis_quality_figures_2026-06-13'
FIGURE_DIR = ARTIFACT_ROOT / 'figures'
METADATA_DIR = ARTIFACT_ROOT / 'metadata'
LATEX_DIR = ARTIFACT_ROOT / 'latex_snippets'

# Qualitative selection. Set SCAN_LIMIT=0 to scan the whole easy/test split.
SCAN_LIMIT = 0
SELECTION_QUANTILES = [0.10, 0.25, 0.50, 0.75, 0.90, 0.98]
SELECTION_METRIC = 'mse'
MAX_SELECTED_SAMPLES = 6
RANDOM_SEED = 20260613

RGB_BANDS = (150, 100, 50)
RGB_PERCENTILES = (1.0, 99.0)
RGB_GAMMA = 1.15
ERROR_PERCENTILE = 99.0
SPECTRUM_PIXEL_MODE = 'max_error'  # 'max_error' or 'center'

# Scan uses forward pass for throughput. Final selected samples can try bitstream mode.
SCAN_USE_BITSTREAM = False
FINAL_USE_BITSTREAM = True
USE_AMP = True
REQUIRE_CUDA = True
FORCE_REINSTALL_ENV = False

# HYPERVIEW2 transfer mosaic inputs. Missing variants are skipped and recorded.
HYPERVIEW2_ROOT = DRIVE_HSI / 'data/hyperview2/HYPERVIEW2'
HYPERVIEW2_ARCHIVE = DRIVE_HSI / 'data/hyperview2/HYPERVIEW2_20260525.tar.gz'
HYPERVIEW2_SAMPLE_IDS = ['0999', '0996', '0998']
HYPERVIEW2_VARIANTS = {
    'Resampling 230->202->230': DRIVE_HSI / 'reconstructions/hyperview2/hyperview2_spectral_resample_passthrough_hyspecnet202_to_230',
    'Mamba K=4 transfer': DRIVE_HSI / 'reconstructions/hyperview2/hyperview2_mamba_k4_spectral_feature_ft_epoch16_hyspecnet202_to_230',
    'CAE 1-D': DRIVE_HSI / 'reconstructions/hyperview2/hyperview2_baseline_1d_pixel_ae_latent1_hyspecnet202_to_230',
}
HYPERVIEW2_RGB_WAVELENGTHS = (650.0, 560.0, 480.0)

print('HySpecNet archive:', HYSPECNET_ARCHIVE)
print('HySpecNet root:', HYSPECNET_ROOT)
print('Checkpoint:', CHECKPOINT_PATH)
print('Artifact root:', ARTIFACT_ROOT)
print('HYPERVIEW2 root:', HYPERVIEW2_ROOT)


## 2. Mount Drive and Prepare Repository

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped or unavailable:', exc)

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)

print('Repo HEAD:')
subprocess.run(['git', 'log', '--oneline', '-1'], cwd=REPO_DIR, check=True)


## 3. Install Runtime Dependencies

The Mamba checkpoint needs the same Torch/Mamba binary stack used by the other Colab notebooks. On the first run this cell installs binary wheels and restarts the runtime; after reconnecting, rerun from the repository cell.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PIP = [sys.executable, '-m', 'pip']
ENV_MARKER = Path('/content/.hsi_compression_hyspecnet_thesis_v2_torch27_mamba232')


def run(cmd, *, required=True):
    print('Running:', ' '.join(map(str, cmd)))
    result = subprocess.run(
        list(map(str, cmd)),
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout[-6000:])
    if result.returncode != 0:
        message = f'Command failed with exit code {result.returncode}: {" ".join(map(str, cmd))}'
        if required:
            raise RuntimeError(message)
        print('Optional command failed:', message)
        return False
    return True


if FORCE_REINSTALL_ENV and ENV_MARKER.exists():
    ENV_MARKER.unlink()

run(['apt-get', 'update'], required=False)
run(['apt-get', 'install', '-y', 'zstd'], required=False)

if not ENV_MARKER.exists():
    run(PIP + ['install', '-q', '--upgrade', 'pip', 'setuptools<82', 'wheel', 'packaging', 'pybind11', 'ninja'])
    run(PIP + [
        'install', '-q', '--force-reinstall',
        'torch==2.7.1', 'torchvision==0.22.1', 'torchaudio==2.7.1',
        '--index-url', 'https://download.pytorch.org/whl/cu126',
    ])
    run(PIP + [
        'install', '-q', '--upgrade', '--force-reinstall',
        'numpy==1.26.4', 'pandas==2.2.2', 'scipy>=1.12,<1.15', 'scikit-learn>=1.6,<1.8',
    ])
    run(PIP + ['install', '-q', '-e', '.[downstream]', 'tqdm', 'matplotlib', 'ipywidgets'])

    import torch
    cxx11_abi = 'TRUE' if getattr(torch._C, '_GLIBCXX_USE_CXX11_ABI', True) else 'FALSE'
    python_tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
    if python_tag != 'cp312':
        raise RuntimeError(f'This prebuilt Mamba preset expects Python 3.12, got {python_tag}.')
    if cxx11_abi != 'TRUE':
        raise RuntimeError(f'This prebuilt Mamba preset expects Torch CXX11 ABI TRUE, got {cxx11_abi}.')
    print('Torch CXX11 ABI:', cxx11_abi)

    run(PIP + [
        'install', '-q', '--force-reinstall', '--no-deps',
        'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.2.post1/causal_conv1d-1.6.2.post1%2Bcu12torch2.7cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
    ])
    run(PIP + [
        'install', '-q', '--force-reinstall', '--no-deps',
        'https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1%2Bcu12torch2.7cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
    ])
    ENV_MARKER.write_text('installed\n', encoding='utf-8')
    print('Dependencies installed. Restarting runtime to reload binary modules.')
    os.kill(os.getpid(), 9)
else:
    print('Dependency marker exists, skipping reinstall:', ENV_MARKER)

import numpy as np
import torch
print('Python:', sys.version)
print('NumPy:', np.__version__)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError('GPU runtime is required. In Colab choose Runtime -> Change runtime type -> GPU.')
if not torch.__version__.startswith('2.7.'):
    if ENV_MARKER.exists():
        ENV_MARKER.unlink()
    raise RuntimeError(
        f'Mamba prebuilt wheels require Torch 2.7.x, but active Torch is {torch.__version__}. '
        'Restart the Colab runtime and rerun from the repo cell.'
    )
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    subprocess.run(['nvidia-smi'], check=False)

try:
    from mamba_ssm import Mamba  # noqa: F401
    print('mamba-ssm import: ok')
except Exception as exc:
    raise RuntimeError(
        'mamba-ssm is required for this notebook. Use a fresh GPU Colab runtime, '
        'set FORCE_REINSTALL_ENV=True, and rerun from the repo cell.'
    ) from exc


## 4. Prepare Data and Output Directories

In [ ]:
import csv
import hashlib
import json
import os
import shutil
import subprocess
import tarfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def count_hyspecnet_files(root: Path) -> int:
    return sum(1 for _ in (root / 'patches').rglob('*-DATA.npy')) if (root / 'patches').exists() else 0


def sha256_file(path: Path, chunk_size: int = 32 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def verify_archive_checksum():
    if not HYSPECNET_ARCHIVE_SHA256.exists():
        print('Checksum file missing; skipping archive SHA-256 verification:', HYSPECNET_ARCHIVE_SHA256)
        return None
    expected = HYSPECNET_ARCHIVE_SHA256.read_text(encoding='utf-8').split()[0]
    actual = sha256_file(HYSPECNET_ARCHIVE)
    if actual != expected:
        raise RuntimeError(f'Archive checksum mismatch: expected {expected}, got {actual}')
    print('Archive SHA-256 verified:', actual)
    return actual


def ensure_hyspecnet_data():
    if not HYSPECNET_ARCHIVE.exists():
        raise FileNotFoundError(f'Missing HySpecNet archive: {HYSPECNET_ARCHIVE}')
    existing = count_hyspecnet_files(HYSPECNET_ROOT)
    if existing == EXPECTED_HYSPECNET_FILES and HYSPECNET_SPLIT.exists():
        print(f'HySpecNet DATA.npy subset already extracted: {HYSPECNET_ROOT} ({existing} files)')
        return
    print(f'Extracting HySpecNet archive to {HYSPECNET_EXTRACT_PARENT}')
    HYSPECNET_EXTRACT_PARENT.mkdir(parents=True, exist_ok=True)
    verify_archive_checksum()
    if HYSPECNET_ROOT.exists():
        shutil.rmtree(HYSPECNET_ROOT)
    subprocess.run(
        ['tar', '-I', 'zstd', '-xf', str(HYSPECNET_ARCHIVE), '-C', str(HYSPECNET_EXTRACT_PARENT)],
        check=True,
    )
    extracted = count_hyspecnet_files(HYSPECNET_ROOT)
    if extracted != EXPECTED_HYSPECNET_FILES:
        raise RuntimeError(f'Expected {EXPECTED_HYSPECNET_FILES} DATA.npy files, found {extracted}')
    if not HYSPECNET_SPLIT.exists():
        raise FileNotFoundError(f'Missing split file after extraction: {HYSPECNET_SPLIT}')
    print(f'Extracted HySpecNet DATA.npy subset: {extracted} files')


def ensure_hyperview2_data():
    if (HYPERVIEW2_ROOT / 'wavelengths.json').exists() and (HYPERVIEW2_ROOT / 'train').exists():
        print('HYPERVIEW2 directory available:', HYPERVIEW2_ROOT)
        return
    if not HYPERVIEW2_ARCHIVE.exists():
        print('HYPERVIEW2 directory/archive missing; mosaic will be skipped:', HYPERVIEW2_ROOT)
        return
    print('Extracting HYPERVIEW2 archive:', HYPERVIEW2_ARCHIVE)
    HYPERVIEW2_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['tar', '-xzf', str(HYPERVIEW2_ARCHIVE), '-C', str(HYPERVIEW2_ROOT.parent)], check=True)


ensure_hyspecnet_data()
ensure_hyperview2_data()
for directory in (FIGURE_DIR, METADATA_DIR, LATEX_DIR):
    directory.mkdir(parents=True, exist_ok=True)

split_entries = []
with HYSPECNET_SPLIT.open(newline='') as f:
    reader = csv.reader(f)
    for row in reader:
        if row and row[0].strip():
            split_entries.append(row[0].strip())

missing = [entry for entry in split_entries if not (HYSPECNET_ROOT / 'patches' / entry).exists()]
if missing:
    raise FileNotFoundError(f'{len(missing)} split entries are missing DATA.npy files. First: {missing[0]}')

print('HySpecNet split entries:', len(split_entries))
print('Artifacts will be written to:', ARTIFACT_ROOT)


## 5. Load Mamba Checkpoint

In [ ]:
import math
from collections import OrderedDict
from pathlib import Path

import torch

from hsi_compression.metrics import masked_mae, masked_mse, masked_psnr, masked_sam_deg, ref_ssim
from hsi_compression.models.registry import build_model

ENTROPY_RUNTIME_KEYS = (
    'entropy_bottleneck._offset',
    'entropy_bottleneck._quantized_cdf',
    'entropy_bottleneck._cdf_length',
)


def load_checkpoint_model(checkpoint_path: Path, in_channels: int, device: torch.device):
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Missing checkpoint: {checkpoint_path}')
    raw = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    cfg = raw.get('config', {})
    model_section = cfg.get('model', {})
    model_name = model_section.get('model_name')
    if not model_name:
        raise ValueError('Checkpoint does not contain config.model.model_name')
    kwargs = dict(model_section.get('model_kwargs', {}))
    kwargs.pop('in_channels', None)

    model = build_model(model_name=model_name, in_channels=in_channels, **kwargs).to(device)
    state = raw.get('model_state_dict') or raw.get('state_dict')
    if state is None:
        raise ValueError('Checkpoint has no model_state_dict/state_dict')
    filtered = OrderedDict(
        (key, value)
        for key, value in state.items()
        if not any(key == runtime_key for runtime_key in ENTROPY_RUNTIME_KEYS)
    )
    missing, unexpected = model.load_state_dict(filtered, strict=False)
    unexpected = list(unexpected)
    not_runtime_missing = [key for key in missing if not any(key == r for r in ENTROPY_RUNTIME_KEYS)]
    if unexpected or not_runtime_missing:
        raise RuntimeError(f'Unexpected load_state_dict result: missing={missing}, unexpected={unexpected}')
    if hasattr(model, 'update'):
        model.update(force=True)
    model.eval()
    return model, cfg, raw

first_cube = np.load(HYSPECNET_ROOT / 'patches' / split_entries[0], mmap_mode='r')
in_channels = int(first_cube.shape[0])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, checkpoint_cfg, raw_checkpoint = load_checkpoint_model(CHECKPOINT_PATH, in_channels, device)
checkpoint_rd_lambda = checkpoint_cfg.get('training', {}).get('rd_lambda')
checkpoint_experiment = checkpoint_cfg.get('experiment', {}).get('name', CHECKPOINT_PATH.stem)

print('Device:', device)
print('Input channels:', in_channels)
print('Model:', checkpoint_cfg.get('model', {}).get('model_name'))
print('Experiment:', checkpoint_experiment)
print('Checkpoint epoch:', raw_checkpoint.get('epoch'))
print('Best val loss:', raw_checkpoint.get('best_val_loss'))
print('rd_lambda from checkpoint config:', checkpoint_rd_lambda)


## 6. Reconstruction, Metrics, and Plot Helpers

In [ ]:
import json
import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'font.family': 'DejaVu Serif',
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'legend.fontsize': 8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'axes.linewidth': 0.8,
})


def rel_to_sample_id(entry: str) -> str:
    return Path(entry).name.removesuffix('-DATA.npy')


def bitstream_num_bytes(strings) -> int:
    if strings is None:
        return 0
    if isinstance(strings, (bytes, bytearray, memoryview)):
        return len(strings)
    if isinstance(strings, str):
        return len(strings.encode('utf-8'))
    if isinstance(strings, dict):
        return sum(bitstream_num_bytes(value) for value in strings.values())
    if isinstance(strings, (list, tuple)):
        return sum(bitstream_num_bytes(value) for value in strings)
    raise TypeError(f'Unsupported bitstream container: {type(strings)!r}')


def _call_forward(x_tensor: torch.Tensor):
    try:
        return model(x_tensor, valid_mask=None)
    except TypeError:
        return model(x_tensor)


@torch.inference_mode()
def reconstruct_cube(x_np: np.ndarray, *, use_bitstream: bool):
    x_tensor = torch.from_numpy(np.ascontiguousarray(x_np)).unsqueeze(0).to(device)
    compressed_bytes = None
    mode_used = 'forward'
    error_message = None
    start = time.perf_counter()

    if use_bitstream and hasattr(model, 'compress') and hasattr(model, 'decompress'):
        try:
            with torch.cuda.amp.autocast(enabled=USE_AMP and device.type == 'cuda'):
                compressed = model.compress(x_tensor)
            strings = compressed.get('strings') if isinstance(compressed, dict) else None
            shape = compressed.get('shape') if isinstance(compressed, dict) else None
            if strings is None or shape is None:
                raise RuntimeError('compress() did not return strings and shape')
            compressed_bytes = bitstream_num_bytes(strings)
            with torch.cuda.amp.autocast(enabled=USE_AMP and device.type == 'cuda'):
                decoded = model.decompress(strings, shape)
            if isinstance(decoded, dict):
                x_hat = decoded.get('x_hat')
            else:
                x_hat = decoded
            if x_hat is None:
                raise RuntimeError('decompress() did not return x_hat')
            mode_used = 'compress_decompress'
            elapsed = time.perf_counter() - start
            return x_hat.detach().float().cpu().squeeze(0).clamp(0.0, 1.0).numpy(), mode_used, elapsed, compressed_bytes, error_message
        except Exception as exc:
            error_message = repr(exc)
            print('Bitstream path failed; falling back to forward pass:', error_message)

    with torch.cuda.amp.autocast(enabled=USE_AMP and device.type == 'cuda'):
        outputs = _call_forward(x_tensor)
    if not isinstance(outputs, dict) or 'x_hat' not in outputs:
        raise RuntimeError('Model forward output must be a dict with x_hat')
    x_hat = outputs['x_hat']
    elapsed = time.perf_counter() - start
    return x_hat.detach().float().cpu().squeeze(0).clamp(0.0, 1.0).numpy(), mode_used, elapsed, compressed_bytes, error_message


def torch_metrics(x_np: np.ndarray, x_hat_np: np.ndarray, compressed_bytes=None) -> dict:
    x = torch.from_numpy(np.ascontiguousarray(x_np)).unsqueeze(0).to(device)
    x_hat = torch.from_numpy(np.ascontiguousarray(x_hat_np)).unsqueeze(0).to(device)
    mask = torch.ones_like(x, dtype=torch.bool)
    with torch.inference_mode():
        mse = float(masked_mse(x_hat, x, mask).detach().cpu().item())
        mae = float(masked_mae(x_hat, x, mask).detach().cpu().item())
        psnr = float(masked_psnr(x_hat, x, mask, data_range=1.0).detach().cpu().item())
        sam = float(masked_sam_deg(x_hat, x, mask).detach().cpu().item())
        try:
            ssim_value = float(ref_ssim(x_hat, x, data_range=1.0, channels=x.shape[1]).detach().cpu().item())
        except Exception as exc:
            print('SSIM computation failed:', repr(exc))
            ssim_value = float('nan')
    bpppc = None if compressed_bytes is None else float(compressed_bytes * 8.0 / x_np.size)
    return {
        'mse': mse,
        'mae': mae,
        'psnr_db': psnr,
        'ssim': ssim_value,
        'sam_deg': sam,
        'bpppc': bpppc,
        'compressed_bytes': compressed_bytes,
        'original_bytes_float32': int(x_np.nbytes),
        'original_bytes_uint16_equivalent': int(x_np.size * 2),
    }


def quick_metrics(x_np: np.ndarray, x_hat_np: np.ndarray) -> dict:
    diff = x_hat_np.astype(np.float32) - x_np.astype(np.float32)
    mse = float(np.mean(diff * diff))
    mae = float(np.mean(np.abs(diff)))
    psnr = float('inf') if mse == 0 else float(10.0 * math.log10(1.0 / max(mse, 1e-12)))
    return {'mse': mse, 'mae': mae, 'psnr_db': psnr}


def rgb_params_for_reference(x: np.ndarray, bands=RGB_BANDS, percentile_range=RGB_PERCENTILES):
    params = []
    for band in bands:
        lo, hi = np.percentile(x[band], percentile_range)
        lo = float(lo)
        hi = float(hi)
        if hi <= lo:
            hi = lo + 1e-8
        params.append((lo, hi))
    return params


def cube_to_rgb(x: np.ndarray, bands=RGB_BANDS, params=None, gamma=RGB_GAMMA):
    if params is None:
        params = rgb_params_for_reference(x, bands=bands)
    channels = []
    for band, (lo, hi) in zip(bands, params):
        channels.append(np.clip((x[band] - lo) / (hi - lo), 0.0, 1.0))
    rgb = np.stack(channels, axis=-1).astype(np.float32)
    if gamma != 1.0:
        rgb = np.power(np.clip(rgb, 0.0, 1.0), 1.0 / gamma)
    return rgb


def error_map(x: np.ndarray, x_hat: np.ndarray) -> np.ndarray:
    return np.mean(np.abs(x_hat - x), axis=0).astype(np.float32)


def choose_spectrum_pixel(x: np.ndarray, x_hat: np.ndarray) -> tuple[int, int]:
    if SPECTRUM_PIXEL_MODE == 'center':
        return int(x.shape[1] // 2), int(x.shape[2] // 2)
    err = error_map(x, x_hat)
    row, col = np.unravel_index(int(np.argmax(err)), err.shape)
    return int(row), int(col)


def select_quantile_records(rows: list[dict], quantiles: list[float], metric: str) -> list[dict]:
    ordered = sorted(rows, key=lambda row: row[metric])
    selected = []
    used_indices = set()
    n = len(ordered)
    for q in quantiles:
        base = int(round(float(q) * (n - 1)))
        for offset in [0, 1, -1, 2, -2, 3, -3]:
            idx = min(max(base + offset, 0), n - 1)
            if idx not in used_indices:
                used_indices.add(idx)
                record = dict(ordered[idx])
                record['selection_quantile'] = float(q)
                record['selection_rank_by_mse'] = int(idx)
                selected.append(record)
                break
        if len(selected) >= MAX_SELECTED_SAMPLES:
            break
    return selected


def write_json(path: Path, payload: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
    print('Saved:', path)


## 7. Scan HySpecNet Easy/Test and Select Qualitative Samples

In [ ]:
scan_entries = split_entries if SCAN_LIMIT in (0, None) else split_entries[: int(SCAN_LIMIT)]
print(f'Scanning {len(scan_entries)} HySpecNet easy/test samples with use_bitstream={SCAN_USE_BITSTREAM}')

scan_rows = []
for split_index, entry in enumerate(tqdm(scan_entries, desc='HySpecNet scan')):
    path = HYSPECNET_ROOT / 'patches' / entry
    x = np.load(path).astype(np.float32)
    x_hat, mode_used, elapsed, compressed_bytes, error_message = reconstruct_cube(x, use_bitstream=SCAN_USE_BITSTREAM)
    metrics = quick_metrics(x, x_hat)
    scan_rows.append({
        'split_index': split_index,
        'sample_id': rel_to_sample_id(entry),
        'split_entry': entry,
        'source_path': str(path),
        'mode_used': mode_used,
        'scan_time_sec': elapsed,
        'scan_error': error_message,
        **metrics,
    })

scan_df = pd.DataFrame(scan_rows)
scan_metrics_path = METADATA_DIR / 'hyspecnet_easy_test_scan_metrics.csv'
scan_df.to_csv(scan_metrics_path, index=False)
print('Saved scan metrics:', scan_metrics_path)

selected_records = select_quantile_records(scan_rows, SELECTION_QUANTILES, SELECTION_METRIC)
selected_df = pd.DataFrame(selected_records)
selected_preview_cols = ['selection_quantile', 'selection_rank_by_mse', 'split_index', 'sample_id', 'mse', 'mae', 'psnr_db']
display(selected_df[selected_preview_cols])


## 8. Generate HySpecNet Figures, Metrics, and Manifest

In [ ]:
selected_payloads = []
metrics_rows = []

for ordinal, selected in enumerate(tqdm(selected_records, desc='Selected HySpecNet samples')):
    entry = selected['split_entry']
    path = HYSPECNET_ROOT / 'patches' / entry
    x = np.load(path).astype(np.float32)
    x_hat, mode_used, elapsed, compressed_bytes, error_message = reconstruct_cube(x, use_bitstream=FINAL_USE_BITSTREAM)
    metrics = torch_metrics(x, x_hat, compressed_bytes=compressed_bytes)
    pixel = choose_spectrum_pixel(x, x_hat)
    err = error_map(x, x_hat)
    record = {
        **selected,
        **metrics,
        'ordinal': ordinal,
        'mode_used_final': mode_used,
        'final_time_sec': elapsed,
        'final_error': error_message,
        'spectrum_pixel_row': pixel[0],
        'spectrum_pixel_col': pixel[1],
        'rgb_bands': list(RGB_BANDS),
        'rgb_percentiles': list(RGB_PERCENTILES),
        'rgb_gamma': RGB_GAMMA,
        'error_map': 'mean absolute error over 202 normalized bands',
    }
    metrics_rows.append(record)
    selected_payloads.append({'record': record, 'x': x, 'x_hat': x_hat, 'error': err})

metrics_df = pd.DataFrame(metrics_rows)
metrics_csv = METADATA_DIR / 'qualitative_samples_metrics.csv'
metrics_df.to_csv(metrics_csv, index=False)
print('Saved metrics:', metrics_csv)
display(metrics_df[['selection_quantile', 'split_index', 'sample_id', 'psnr_db', 'ssim', 'sam_deg', 'mae', 'bpppc', 'mode_used_final']])

# Equal-size qualitative grid: original, reconstruction, error map.
n = len(selected_payloads)
fig = plt.figure(figsize=(10.8, max(3.0, 3.0 * n)))
grid = fig.add_gridspec(nrows=n, ncols=4, width_ratios=[1.0, 1.0, 1.0, 0.045], wspace=0.08, hspace=0.26)
error_vmax = float(np.nanpercentile(np.concatenate([item['error'].ravel() for item in selected_payloads]), ERROR_PERCENTILE))
if not np.isfinite(error_vmax) or error_vmax <= 0:
    error_vmax = max(float(np.nanmax([np.nanmax(item['error']) for item in selected_payloads])), 1e-6)

for row, item in enumerate(selected_payloads):
    rec = item['record']
    x = item['x']
    x_hat = item['x_hat']
    err = item['error']
    rgb_params = rgb_params_for_reference(x)
    original_rgb = cube_to_rgb(x, params=rgb_params)
    recon_rgb = cube_to_rgb(x_hat, params=rgb_params)

    axes = [fig.add_subplot(grid[row, col]) for col in range(3)]
    cax = fig.add_subplot(grid[row, 3])
    for ax in axes:
        ax.set_box_aspect(1)
        ax.set_xticks([])
        ax.set_yticks([])
    axes[0].imshow(original_rgb, interpolation='nearest')
    axes[1].imshow(recon_rgb, interpolation='nearest')
    im = axes[2].imshow(err, cmap='magma', vmin=0.0, vmax=error_vmax, interpolation='nearest')
    fig.colorbar(im, cax=cax)
    if row == 0:
        axes[0].set_title('Oryginał')
        axes[1].set_title('Rekonstrukcja Mamba')
        axes[2].set_title('Mapa błędu MAE')
    label = f"q={rec['selection_quantile']:.2f}, PSNR={rec['psnr_db']:.2f} dB, SAM={rec['sam_deg']:.2f}°"
    axes[0].set_ylabel(label, rotation=90, labelpad=16)

fig.suptitle('HySpecNet-11k easy/test: Mamba K=4 RD qualitative examples', y=0.995)
grid_png = FIGURE_DIR / 'hyspecnet_mamba_k4_rd_qualitative_grid.png'
grid_pdf = FIGURE_DIR / 'hyspecnet_mamba_k4_rd_qualitative_grid.pdf'
fig.savefig(grid_png, dpi=300, bbox_inches='tight')
fig.savefig(grid_pdf, bbox_inches='tight')
plt.close(fig)
print('Saved:', grid_png)
print('Saved:', grid_pdf)

# Spectral signatures figure.
fig, axes = plt.subplots(n, 1, figsize=(10.8, max(2.4, 2.25 * n)), squeeze=False)
for row, item in enumerate(selected_payloads):
    rec = item['record']
    x = item['x']
    x_hat = item['x_hat']
    r = int(rec['spectrum_pixel_row'])
    c = int(rec['spectrum_pixel_col'])
    ax = axes[row, 0]
    bands = np.arange(x.shape[0])
    ax.plot(bands, x[:, r, c], color='black', linewidth=1.8, label=f'oryginał ({r}, {c})')
    ax.plot(bands, x_hat[:, r, c], color='#d55e00', linewidth=1.5, label='rekonstrukcja')
    ax.plot(bands, x.mean(axis=(1, 2)), color='black', linewidth=1.0, alpha=0.35, linestyle=':', label='średnia oryginału')
    ax.plot(bands, x_hat.mean(axis=(1, 2)), color='#d55e00', linewidth=1.0, alpha=0.45, linestyle=':', label='średnia rekonstrukcji')
    ax.set_title(f"{rec['sample_id']} | q={rec['selection_quantile']:.2f} | PSNR={rec['psnr_db']:.2f} dB")
    ax.set_xlabel('Pasmo spektralne po usunięciu pasm absorpcji pary wodnej')
    ax.set_ylabel('Znormalizowana reflektancja')
    ax.grid(alpha=0.25)
    ax.legend(loc='best', ncols=2)
fig.tight_layout()
spectra_png = FIGURE_DIR / 'hyspecnet_mamba_k4_rd_spectra.png'
spectra_pdf = FIGURE_DIR / 'hyspecnet_mamba_k4_rd_spectra.pdf'
fig.savefig(spectra_png, dpi=300, bbox_inches='tight')
fig.savefig(spectra_pdf, bbox_inches='tight')
plt.close(fig)
print('Saved:', spectra_png)
print('Saved:', spectra_pdf)

# Save compact reconstructions for later reuse without rerunning the model.
reconstruction_dir = ARTIFACT_ROOT / 'reconstructions'
reconstruction_dir.mkdir(parents=True, exist_ok=True)
for item in selected_payloads:
    rec = item['record']
    np.savez_compressed(
        reconstruction_dir / f"{rec['ordinal']:02d}_{rec['sample_id']}_mamba_k4_rd.npz",
        original=item['x'],
        reconstruction=item['x_hat'],
        error=item['error'],
    )


## 9. Generate HYPERVIEW2 Transfer Mosaic

In [ ]:
def resolve_hv2_root(root: Path) -> Path:
    if (root / 'HYPERVIEW2').is_dir():
        return root / 'HYPERVIEW2'
    return root


def hv2_sample_stems(sample_id: str) -> list[str]:
    value = str(sample_id).strip()
    stems = [value]
    if value.isdigit():
        stems.append(f'{int(value):04d}')
    return list(dict.fromkeys(stems))


def find_hv2_sample(root: Path, sample_id: str) -> Path | None:
    root = resolve_hv2_root(root)
    candidates = []
    for split in ('train', 'test'):
        for kind in ('hsi_satellite', 'msi_satellite'):
            for stem in hv2_sample_stems(sample_id):
                candidates.append(root / split / kind / f'{stem}.npz')
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def to_chw(array: np.ndarray) -> np.ndarray:
    array = np.asarray(array)
    while array.ndim > 3:
        singleton_axes = [axis for axis, dim in enumerate(array.shape) if dim == 1]
        if not singleton_axes:
            break
        array = np.squeeze(array, axis=singleton_axes[0])
    if array.ndim != 3:
        raise ValueError(f'Expected 3D cube, got shape={array.shape}')
    if array.shape[0] <= 512:
        return np.ascontiguousarray(array, dtype=np.float32)
    return np.ascontiguousarray(np.moveaxis(array, -1, 0), dtype=np.float32)


def load_hv2_npz(path: Path) -> tuple[np.ndarray, np.ndarray | None]:
    with np.load(path) as archive:
        if 'data' in archive.files:
            cube = to_chw(archive['data'])
        else:
            cubes = [archive[key] for key in archive.files if archive[key].ndim >= 3]
            if not cubes:
                raise ValueError(f'No cube found in {path}')
            cube = to_chw(max(cubes, key=lambda arr: arr.size))
        mask = np.asarray(archive['mask']) if 'mask' in archive.files else None
    if mask is None:
        return cube, None
    while mask.ndim > 3:
        singleton_axes = [axis for axis, dim in enumerate(mask.shape) if dim == 1]
        if not singleton_axes:
            break
        mask = np.squeeze(mask, axis=singleton_axes[0])
    if mask.ndim == 3:
        if mask.shape == cube.shape:
            mask = mask.any(axis=0)
        elif mask.shape[-2:] == cube.shape[-2:]:
            mask = mask.any(axis=0)
        elif mask.shape[:2] == cube.shape[-2:]:
            mask = mask.any(axis=-1)
    if mask.ndim != 2 or mask.shape != cube.shape[-2:]:
        print('Ignoring incompatible mask shape:', mask.shape, 'for', path)
        return cube, None
    return cube, mask.astype(bool)


def load_hv2_wavelengths(root: Path, channels: int) -> np.ndarray:
    path = resolve_hv2_root(root) / 'wavelengths.json'
    if not path.exists():
        return np.arange(channels, dtype=np.float32)
    payload = json.loads(path.read_text(encoding='utf-8'))
    values = payload.get('hsi_satellite_wavelengths', payload) if isinstance(payload, dict) else payload
    if isinstance(values, dict):
        return np.asarray([float(values.get(f'Band {idx}', idx)) for idx in range(channels)], dtype=np.float32)
    arr = np.asarray(values, dtype=np.float32).reshape(-1)
    return arr if len(arr) == channels else np.arange(channels, dtype=np.float32)


def hv2_rgb_bands(wavelengths: np.ndarray, targets=HYPERVIEW2_RGB_WAVELENGTHS) -> tuple[int, int, int]:
    if len(wavelengths) == 0:
        return (0, 0, 0)
    bands = tuple(int(np.argmin(np.abs(wavelengths - target))) for target in targets)
    if len(set(bands)) == 3:
        return bands
    c = len(wavelengths)
    return (min(c - 1, int(c * 0.62)), min(c - 1, int(c * 0.38)), min(c - 1, int(c * 0.18)))


def hv2_spatial_mask(mask: np.ndarray | None, cube: np.ndarray) -> np.ndarray:
    if mask is None:
        return np.isfinite(cube).all(axis=0)
    return mask.astype(bool) & np.isfinite(cube).all(axis=0)


def hv2_rgb(cube: np.ndarray, bands: tuple[int, int, int], params=None, spatial_mask=None):
    if spatial_mask is None:
        spatial_mask = np.isfinite(cube).all(axis=0)
    if params is None:
        params = []
        for band in bands:
            values = cube[band][spatial_mask]
            if values.size == 0:
                params.append((0.0, 1.0))
            else:
                lo, hi = np.percentile(values, [2.0, 98.0])
                if hi <= lo:
                    hi = lo + 1e-8
                params.append((float(lo), float(hi)))
    rgb = []
    for band, (lo, hi) in zip(bands, params):
        rgb.append(np.clip((cube[band] - lo) / (hi - lo), 0.0, 1.0))
    out = np.stack(rgb, axis=-1).astype(np.float32)
    out[~spatial_mask] = 0.0
    return out, params


hv2_records = []
if not HYPERVIEW2_ROOT.exists():
    print('Skipping HYPERVIEW2 mosaic because root is missing:', HYPERVIEW2_ROOT)
else:
    first_path = find_hv2_sample(HYPERVIEW2_ROOT, HYPERVIEW2_SAMPLE_IDS[0])
    if first_path is None:
        print('Skipping HYPERVIEW2 mosaic; first sample missing:', HYPERVIEW2_SAMPLE_IDS[0])
    else:
        first_cube, _ = load_hv2_npz(first_path)
        hv2_wavelengths = load_hv2_wavelengths(HYPERVIEW2_ROOT, first_cube.shape[0])
        hv2_bands = hv2_rgb_bands(hv2_wavelengths)
        columns = ['Oryginał', *HYPERVIEW2_VARIANTS.keys(), 'Maska']
        fig, axes = plt.subplots(
            len(HYPERVIEW2_SAMPLE_IDS),
            len(columns),
            figsize=(2.6 * len(columns), 2.7 * len(HYPERVIEW2_SAMPLE_IDS)),
            squeeze=False,
        )
        for row, sample_id in enumerate(HYPERVIEW2_SAMPLE_IDS):
            source_path = find_hv2_sample(HYPERVIEW2_ROOT, sample_id)
            if source_path is None:
                for ax in axes[row]:
                    ax.set_axis_off()
                hv2_records.append({'sample_id': sample_id, 'status': 'missing_original'})
                continue
            original, mask = load_hv2_npz(source_path)
            spatial_mask = hv2_spatial_mask(mask, original)
            original_rgb, rgb_params = hv2_rgb(original, hv2_bands, spatial_mask=spatial_mask)
            axes[row, 0].imshow(original_rgb, interpolation='nearest')
            axes[row, 0].set_title('Oryginał' if row == 0 else '')
            axes[row, 0].set_ylabel(str(sample_id))
            axes[row, 0].set_xticks([])
            axes[row, 0].set_yticks([])
            sample_record = {
                'sample_id': sample_id,
                'source_path': str(source_path),
                'mask_available': mask is not None,
                'variants': {},
            }
            col = 1
            for label, root in HYPERVIEW2_VARIANTS.items():
                recon_path = find_hv2_sample(root, sample_id)
                ax = axes[row, col]
                if recon_path is None:
                    ax.text(0.5, 0.5, 'brak', ha='center', va='center')
                    ax.set_axis_off()
                    sample_record['variants'][label] = {'status': 'missing', 'root': str(root)}
                else:
                    recon, _ = load_hv2_npz(recon_path)
                    if recon.shape != original.shape:
                        ax.text(0.5, 0.5, f'shape\n{recon.shape}', ha='center', va='center')
                        ax.set_axis_off()
                        sample_record['variants'][label] = {'status': 'shape_mismatch', 'path': str(recon_path), 'shape': list(recon.shape)}
                    else:
                        recon_rgb, _ = hv2_rgb(recon, hv2_bands, params=rgb_params, spatial_mask=spatial_mask)
                        mae = float(np.mean(np.abs(recon - original)))
                        ax.imshow(recon_rgb, interpolation='nearest')
                        ax.set_xticks([])
                        ax.set_yticks([])
                        sample_record['variants'][label] = {'status': 'ok', 'path': str(recon_path), 'mae': mae}
                if row == 0:
                    ax.set_title(label)
                col += 1
            mask_ax = axes[row, col]
            if mask is None:
                mask_ax.text(0.5, 0.5, 'brak maski', ha='center', va='center')
                mask_ax.set_axis_off()
            else:
                mask_ax.imshow(mask.astype(np.float32), cmap='gray', vmin=0, vmax=1, interpolation='nearest')
                mask_ax.set_xticks([])
                mask_ax.set_yticks([])
            if row == 0:
                mask_ax.set_title('Maska')
            hv2_records.append(sample_record)
        fig.suptitle('HYPERVIEW2: oryginał, resampling, transfer Mamba, CAE 1-D i maski', y=0.995)
        fig.tight_layout()
        hv2_png = FIGURE_DIR / 'hyperview2_transfer_mosaic.png'
        hv2_pdf = FIGURE_DIR / 'hyperview2_transfer_mosaic.pdf'
        fig.savefig(hv2_png, dpi=300, bbox_inches='tight')
        fig.savefig(hv2_pdf, bbox_inches='tight')
        plt.close(fig)
        print('Saved:', hv2_png)
        print('Saved:', hv2_pdf)


## 10. Metadata, Environment, and LaTeX Snippets

In [ ]:
def rel_artifact(path: Path) -> str:
    try:
        return str(path.relative_to(DRIVE_HSI))
    except ValueError:
        return str(path)


def tex_escape(value: str) -> str:
    replacements = {
        '&': r'\&', '%': r'\%', '$': r'\$', '#': r'\#', '_': r'\_',
        '{': r'\{', '}': r'\}', '~': r'\textasciitilde{}', '^': r'\textasciicircum{}',
        '\\': r'\textbackslash{}',
    }
    return ''.join(replacements.get(char, char) for char in str(value))

repo_hash = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True, capture_output=True, check=True).stdout.strip()
python_version = sys.version.replace('\n', ' ')
package_versions = {
    'python': python_version,
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'torch': torch.__version__,
    'cuda_available': bool(torch.cuda.is_available()),
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
try:
    import matplotlib
    package_versions['matplotlib'] = matplotlib.__version__
except Exception:
    pass
try:
    import mamba_ssm
    package_versions['mamba_ssm'] = getattr(mamba_ssm, '__version__', 'installed')
except Exception as exc:
    package_versions['mamba_ssm'] = f'unavailable: {exc!r}'

created_at = datetime.now(timezone.utc).isoformat()
manifest_payload = {
    'created_at': created_at,
    'artifact_root': str(ARTIFACT_ROOT),
    'repository': str(REPO_DIR),
    'git_commit': repo_hash,
    'dataset': {
        'name': 'HySpecNet-11k',
        'split': 'easy/test',
        'root': str(HYSPECNET_ROOT),
        'archive': str(HYSPECNET_ARCHIVE),
        'split_csv': str(HYSPECNET_SPLIT),
        'split_entries': len(split_entries),
        'protocol': 'benchmark DATA.npy paths relative to patches/',
        'normalization': 'TIF nodata -32768 -> 0, water-vapor bands removed, clipped to 0..10000, divided by 10000, float32 (202,128,128)',
    },
    'checkpoint': {
        'path': str(CHECKPOINT_PATH),
        'experiment': checkpoint_experiment,
        'rd_lambda': checkpoint_rd_lambda,
        'epoch': raw_checkpoint.get('epoch'),
        'best_val_loss': raw_checkpoint.get('best_val_loss'),
        'model_name': checkpoint_cfg.get('model', {}).get('model_name'),
        'model_kwargs': checkpoint_cfg.get('model', {}).get('model_kwargs'),
    },
    'selection': {
        'metric': SELECTION_METRIC,
        'quantiles': SELECTION_QUANTILES,
        'scan_limit': SCAN_LIMIT,
        'scan_use_bitstream': SCAN_USE_BITSTREAM,
        'final_use_bitstream': FINAL_USE_BITSTREAM,
        'random_seed': RANDOM_SEED,
        'deterministic': 'samples selected by sorted reconstruction MSE quantiles; no random sampling used',
    },
    'visualization': {
        'pseudo_rgb_bands': list(RGB_BANDS),
        'contrast_stretch_percentiles': list(RGB_PERCENTILES),
        'gamma': RGB_GAMMA,
        'error_map': 'mean absolute error over all normalized spectral bands; shared color scale uses percentile cap',
        'spectrum_pixel_mode': SPECTRUM_PIXEL_MODE,
    },
    'artifacts': {
        'hyspecnet_grid_png': rel_artifact(FIGURE_DIR / 'hyspecnet_mamba_k4_rd_qualitative_grid.png'),
        'hyspecnet_grid_pdf': rel_artifact(FIGURE_DIR / 'hyspecnet_mamba_k4_rd_qualitative_grid.pdf'),
        'hyspecnet_spectra_png': rel_artifact(FIGURE_DIR / 'hyspecnet_mamba_k4_rd_spectra.png'),
        'hyspecnet_spectra_pdf': rel_artifact(FIGURE_DIR / 'hyspecnet_mamba_k4_rd_spectra.pdf'),
        'hyperview2_mosaic_png': rel_artifact(FIGURE_DIR / 'hyperview2_transfer_mosaic.png'),
        'hyperview2_mosaic_pdf': rel_artifact(FIGURE_DIR / 'hyperview2_transfer_mosaic.pdf'),
        'metrics_csv': rel_artifact(metrics_csv),
        'scan_metrics_csv': rel_artifact(scan_metrics_path),
    },
    'samples': metrics_rows,
    'hyperview2': {
        'root': str(HYPERVIEW2_ROOT),
        'sample_ids': HYPERVIEW2_SAMPLE_IDS,
        'variants': {label: str(root) for label, root in HYPERVIEW2_VARIANTS.items()},
        'records': hv2_records,
    },
    'warnings': [
        'HySpecNet qualitative sample selection uses forward-pass MSE for scanning; final bpppc is present only when compress/decompress succeeds.',
        'HYPERVIEW2 mosaic uses saved reconstructions from Drive and is diagnostic, not a HySpecNet reference-comparable result.',
    ],
}
manifest_json = METADATA_DIR / 'qualitative_samples_manifest.json'
write_json(manifest_json, manifest_payload)

# Environment report.
env_lines = [
    '# Generation Environment',
    '',
    f'- Generated at: `{created_at}`',
    f'- Repository path: `{REPO_DIR}`',
    f'- Git commit: `{repo_hash}`',
    f'- HySpecNet archive: `{HYSPECNET_ARCHIVE}`',
    f'- HySpecNet dataset root: `{HYSPECNET_ROOT}`',
    f'- Checkpoint: `{CHECKPOINT_PATH}`',
    f'- Artifact root: `{ARTIFACT_ROOT}`',
    f'- GPU used: `{torch.cuda.is_available()}`',
]
if torch.cuda.is_available():
    env_lines.append(f'- GPU name: `{torch.cuda.get_device_name(0)}`')
env_lines += [
    '',
    '## Package Versions',
    '',
]
for key, value in package_versions.items():
    env_lines.append(f'- `{key}`: `{value}`')
env_lines += [
    '',
    '## Commands',
    '',
    '```bash',
    f'tar -I zstd -xf {HYSPECNET_ARCHIVE} -C {HYSPECNET_EXTRACT_PARENT}',
    '# Then run this notebook top-to-bottom in a GPU Colab runtime.',
    '```',
    '',
    '## Limitations',
    '',
    '- The HYPERVIEW2 panel is diagnostic and uses saved reconstructions; it is not a HySpecNet benchmark result.',
    '- `bpppc` is recorded only when the checkpoint bitstream path succeeds for selected samples.',
]
env_md = METADATA_DIR / 'generation_environment.md'
env_md.write_text('\n'.join(env_lines) + '\n', encoding='utf-8')
print('Saved:', env_md)

# LaTeX captions and include snippets.
captions = r'''
% Generated captions for thesis figures.
\newcommand{\captionHyspecnetMambaQualitative}{Jakościowe przykłady rekonstrukcji próbek ze zbioru HySpecNet-11k easy/test dla hierarchicznego modelu Mamba z parametrem $K=4$ i punktem RD odpowiadającym $\lambda \approx 3\cdot 10^{-4}$. W kolejnych kolumnach pokazano kompozycję pseudo-RGB oryginału, rekonstrukcję oraz mapę średniego błędu bezwzględnego licznego po pasmach spektralnych. Wszystkie panele obrazowe mają ten sam rozmiar wizualny; skala barw mapy błędu jest wspólna dla próbek. Źródło: opracowanie własne na podstawie danych HySpecNet-11k.}
\newcommand{\captionHyspecnetMambaSpectra}{Porównanie sygnatur spektralnych oryginału i rekonstrukcji dla wybranych próbek HySpecNet-11k easy/test. Piksele wybrano deterministycznie według największego błędu rekonstrukcji w danej próbce; linie kropkowane pokazują średnią sygnaturę dla całej próbki. Źródło: opracowanie własne na podstawie danych HySpecNet-11k.}
\newcommand{\captionHyperviewTransferMosaic}{Mozaika diagnostyczna HYPERVIEW2 przedstawiająca przykładowe próbki oryginalne oraz dostępne warianty przetwarzania: resampling spektralny $230\rightarrow202\rightarrow230$, rekonstrukcję modelu Mamba oraz baseline CAE 1-D. Ostatnia kolumna pokazuje maski dostępne w zbiorze. Figura ma charakter diagnostyczny dla transferu między zbiorami. Źródło: opracowanie własne na podstawie danych HYPERVIEW2.}
'''.strip() + '\n'
captions_path = LATEX_DIR / 'figure_captions_pl.tex'
captions_path.write_text(captions, encoding='utf-8')
print('Saved:', captions_path)

snippets = r'''
\begin{figure}[htbp]
\centering
\includegraphics[width=\textwidth]{remote_artifacts/thesis_quality_figures_2026-06-13/figures/hyspecnet_mamba_k4_rd_qualitative_grid.pdf}
\caption[Przykłady jakościowe rekonstrukcji HySpecNet-11k dla modelu Mamba]{\captionHyspecnetMambaQualitative}
\label{fig:hyspecnet-mamba-k4-rd-qualitative-grid}
\end{figure}

\begin{figure}[htbp]
\centering
\includegraphics[width=\textwidth]{remote_artifacts/thesis_quality_figures_2026-06-13/figures/hyspecnet_mamba_k4_rd_spectra.pdf}
\caption[Sygnatury spektralne rekonstrukcji HySpecNet-11k dla modelu Mamba]{\captionHyspecnetMambaSpectra}
\label{fig:hyspecnet-mamba-k4-rd-spectra}
\end{figure}

\begin{figure}[htbp]
\centering
\includegraphics[width=\textwidth]{remote_artifacts/thesis_quality_figures_2026-06-13/figures/hyperview2_transfer_mosaic.pdf}
\caption[Mozaika diagnostyczna transferu HYPERVIEW2]{\captionHyperviewTransferMosaic}
\label{fig:hyperview2-transfer-mosaic}
\end{figure}
'''.strip() + '\n'
snippets_path = LATEX_DIR / 'include_snippets.tex'
snippets_path.write_text(snippets, encoding='utf-8')
print('Saved:', snippets_path)

# Final report.
expected_files = [
    FIGURE_DIR / 'hyspecnet_mamba_k4_rd_qualitative_grid.pdf',
    FIGURE_DIR / 'hyspecnet_mamba_k4_rd_qualitative_grid.png',
    FIGURE_DIR / 'hyspecnet_mamba_k4_rd_spectra.pdf',
    FIGURE_DIR / 'hyspecnet_mamba_k4_rd_spectra.png',
    FIGURE_DIR / 'hyperview2_transfer_mosaic.pdf',
    FIGURE_DIR / 'hyperview2_transfer_mosaic.png',
    METADATA_DIR / 'qualitative_samples_manifest.json',
    METADATA_DIR / 'qualitative_samples_metrics.csv',
    METADATA_DIR / 'generation_environment.md',
    LATEX_DIR / 'figure_captions_pl.tex',
    LATEX_DIR / 'include_snippets.tex',
]
print('\nFinal artifact status:')
for file_path in expected_files:
    print(f'- {file_path}:', 'OK' if file_path.exists() else 'MISSING')

print('\nSource checkpoint:', CHECKPOINT_PATH)
print('Source HySpecNet archive:', HYSPECNET_ARCHIVE)
print('Recreate: run this notebook top-to-bottom after mounting Google Drive.')
